In [46]:
import os
os.environ['KERAS_BACKEND'] = 'torch'

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import keras
from keras.datasets import cifar10
from keras.utils import plot_model

from xgboost import XGBClassifier

from sklearn.metrics import classification_report, confusion_matrix

import utils
import models
from importlib import reload
reload(utils)
reload(models)

from utils import *
from models import XgboostCNN

In [51]:
from keras import applications


<module 'keras.api._v2.keras.applications.efficientnet' from 'c:\\Users\\romain\\anaconda3\\envs\\xgb_cpu\\lib\\site-packages\\keras\\api\\_v2\\keras\\applications\\efficientnet\\__init__.py'>

# XGBOOST with resnet 50

cnn as feature extractor for high N class and low sample use case.

In [43]:
INPUT_SHAPE = (32, 32, 3)
SAMPLE_SIZE = 5000

In [44]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

sample = np.random.randint(0, x_train.shape[0], SAMPLE_SIZE)
x_train = x_train[sample]
y_train = y_train[sample]
x_train.shape

(5000, 32, 32, 3)

In [48]:
target_shape = (50, 50, 3)
model = XgboostCNN(INPUT_SHAPE, target_shape, base_model='RESNET50')

model.fit(x_train, y_train)

model.extractor.summary()

extracting features map
157/157 [==============================] - 33s 202ms/step
Fitting xgb


c:\Users\romain\anaconda3\envs\xgb_cpu\lib\site-packages\xgboost\sklearn.py:1224: UserWarning: The use of label encoder in XGBClassifier is deprecated and will be removed in a future release. To remove this warning, do the following: 1) Pass option use_label_encoder=False when constructing XGBClassifier object; and 2) Encode your labels (y) as integers starting with 0, i.e. 0, 1, 2, ..., [num_class - 1].
  warnings.warn(label_encoder_deprecation_msg, UserWarning)
c:\Users\romain\anaconda3\envs\xgb_cpu\lib\site-packages\sklearn\preprocessing\_label.py:98: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
c:\Users\romain\anaconda3\envs\xgb_cpu\lib\site-packages\sklearn\preprocessing\_label.py:133: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().


ValueError: Please reshape the input data into 2-dimensional matrix.

In [11]:
fmap = model.fmap_train_
fmap[1].shape

(1000, 13, 13, 64)

# Benchmark

For benchmarking use cifar100 instead of cifar10

<br>

In [22]:
y_pred = model.predict(x_test)

print(classification_report(y_test, y_pred))

313/313 [==============================] - 55s 177ms/step
              precision    recall  f1-score   support

           0       0.64      0.58      0.61      1000
           1       0.72      0.73      0.73      1000
           2       0.55      0.50      0.52      1000
           3       0.45      0.48      0.46      1000
           4       0.55      0.48      0.52      1000
           5       0.58      0.60      0.59      1000
           6       0.65      0.66      0.66      1000
           7       0.61      0.65      0.63      1000
           8       0.65      0.67      0.66      1000
           9       0.66      0.72      0.69      1000

    accuracy                           0.61     10000
   macro avg       0.61      0.61      0.61     10000
weighted avg       0.61      0.61      0.61     10000

